In [1]:
from langchain_deepseek import ChatDeepSeek
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import MessagesState
from langchain.messages import HumanMessage

from dotenv import load_dotenv
load_dotenv(override=True)

model = ChatDeepSeek(
    model='deepseek-v4-flash',
    extra_body={
        "thinking": {
            "type": "disabled"
        }
    }
)

class OverAllState(MessagesState):
    username: str
    output: str

def node_a(state: OverAllState) -> OverAllState:
    return {
        "messages": [HumanMessage("你好，我是 " + state["username"])]
    }

def llm_node(state: OverAllState) -> OverAllState:
    res = model.invoke(state["messages"])

    return {
        "messages": [res],
        "output": res.content
    }

builder = StateGraph(state_schema=OverAllState)
builder.add_node("node_a", node_a)
builder.add_node("llm_node", llm_node)
builder.add_edge(START, "node_a")
builder.add_edge("node_a", "llm_node")
builder.add_edge("llm_node", END)

graph = builder.compile()
response = graph.invoke({"username": "小黄"})
print(response)

{'messages': [HumanMessage(content='你好，我是 小黄', additional_kwargs={}, response_metadata={}, id='07c7c8a4-f752-4c8a-8463-5db9532f3a74'), AIMessage(content='你好呀，小黄！很高兴认识你 😊\n\n有什么我可以帮你的吗？无论是聊天、解答问题、帮忙写点东西，还是其他什么，都可以跟我说～', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 10, 'total_tokens': 46, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 10}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669', 'id': 'ecf9c1f2-3144-4298-b5b5-563d633dfb8a', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0a3d2-5448-7340-8374-071c0d517015-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 10, 'output_tokens': 36, 'total_tokens': 46, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}})], 'username': '小黄', 'output